In [21]:
import os
import gc
import argparse
import shutil
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
import pyarrow as pa
import duckdb

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def reset_dir(path: str):
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)


def connect_duckdb(threads=2, memory_limit="180GB", temp_dir="./tmp_duckdb"):
    # Ensure the temp directory exists before pointing DuckDB to it
    os.makedirs(temp_dir, exist_ok=True) 
    
    con = duckdb.connect()
    con.execute(f"SET threads TO {int(threads)};")
    con.execute(f"SET memory_limit = '{memory_limit}';")
    con.execute(f"SET temp_directory = '{temp_dir}';")
    return con


# -----------------------------------------------------------------------------
# Step 1: Spatial mapping (sensor → nearest station)
# -----------------------------------------------------------------------------

def build_nearest_station_mapping(
    con,
    flood_file,
    precip_master_file,
    sensors_crs="EPSG:4326",
    planar_crs="EPSG:2263",
):
    print("\n[1/3] Extracting unique sensors...")

    sensors = con.execute(f"""
        SELECT DISTINCT deployment_id, latitude, longitude
        FROM '{flood_file}'
        WHERE deployment_id IS NOT NULL
          AND latitude IS NOT NULL
          AND longitude IS NOT NULL
    """).df()

    if sensors.empty:
        raise RuntimeError("No valid sensors found.")

    df_precip = pd.read_parquet(precip_master_file)

    stations = (
        df_precip[["STATION", "LATITUDE", "LONGITUDE"]]
        .dropna()
        .drop_duplicates()
        .reset_index(drop=True)
    )

    gdf_sensors = gpd.GeoDataFrame(
        sensors,
        geometry=gpd.points_from_xy(
            sensors.longitude, sensors.latitude
        ),
        crs=sensors_crs,
    ).to_crs(planar_crs)

    gdf_stations = gpd.GeoDataFrame(
        stations,
        geometry=gpd.points_from_xy(
            stations.LONGITUDE, stations.LATITUDE
        ),
        crs=sensors_crs,
    ).to_crs(planar_crs)

    mapping = gpd.sjoin_nearest(
        gdf_sensors,
        gdf_stations[["STATION", "geometry"]],
        how="left",
        distance_col="dist_to_station_ft",
    )

    out = (
        mapping[["deployment_id", "STATION", "dist_to_station_ft"]]
        .dropna()
        .reset_index(drop=True)
    )

    print(
        f"[1/3] ✅ {out['deployment_id'].nunique()} sensors → "
        f"{out['STATION'].nunique()} stations"
    )

    return out


# -----------------------------------------------------------------------------
# Step 2: Temporal join (schema-safe)
# -----------------------------------------------------------------------------

def temporal_join_per_sensor(
    con,
    flood_file,
    precip_master_file,
    mapping_df,
    output_file,
    chunk_dir,
    asof_tolerance_min=15,
    compression="zstd",
):
    print("\n[2/3] Loading precipitation master...")
    df_precip = pd.read_parquet(precip_master_file)

    df_precip["DATE"] = pd.to_datetime(
        df_precip["DATE"], utc=True, errors="coerce"
    )
    df_precip = df_precip.dropna(subset=["DATE", "STATION"])

    df_precip_clean = df_precip.drop(
        columns=[c for c in ["LATITUDE", "LONGITUDE"] if c in df_precip],
        errors="ignore",
    )

    # ✅ Canonical schema
    weather_schema = (
        df_precip_clean
        .drop(columns=["STATION"])
        .dtypes
        .to_dict()
    )

    station_weather = {}
    for st, grp in df_precip_clean.groupby("STATION", sort=False):
        station_weather[st] = (
            grp.sort_values("DATE")
            .drop(columns=["STATION"])
            .reset_index(drop=True)
        )

    reset_dir(chunk_dir)

    sensor_to_station = dict(zip(
        mapping_df["deployment_id"], mapping_df["STATION"]
    ))
    sensor_to_dist = dict(zip(
        mapping_df["deployment_id"],
        mapping_df["dist_to_station_ft"],
    ))

    tol = pd.Timedelta(minutes=asof_tolerance_min)

    print(f"[2/3] Temporal join ({len(sensor_to_station)} sensors)")

    for i, sensor_id in enumerate(sensor_to_station, start=1):
        if i == 1 or i % 25 == 0:
            print(f"   [{i}] {sensor_id}")

        df_sensor = con.execute(f"""
            SELECT *
            FROM '{flood_file}'
            WHERE deployment_id = '{sensor_id}'
        """).df()

        if df_sensor.empty:
            continue

        df_sensor["time"] = pd.to_datetime(
            df_sensor["time"], utc=True, errors="coerce"
        )
        df_sensor = df_sensor.dropna(subset=["time"]).sort_values("time")

        station = sensor_to_station[sensor_id]
        wx = station_weather.get(station)
        if wx is None:
            continue

        df_sensor["STATION"] = station
        df_sensor["dist_to_station_ft"] = sensor_to_dist[sensor_id]

        df_joined = pd.merge_asof(
            df_sensor,
            wx,
            left_on="time",
            right_on="DATE",
            direction="backward",
            tolerance=tol,
        )

        df_joined = (
            df_joined.dropna(subset=["DATE"])
            .rename(columns={"DATE": "weather_time"})
            .reset_index(drop=True)
        )

        # ✅ Force schema
        for col, dtype in weather_schema.items():
            if col not in df_joined:
                df_joined[col] = pd.Series(dtype=dtype)
            else:
                df_joined[col] = df_joined[col].astype(dtype, errors="ignore")

        df_joined["minutes_since_weather_update"] = (
            (df_joined["time"] - df_joined["weather_time"])
            .dt.total_seconds() / 60.0
        )

        out_path = os.path.join(
            chunk_dir, f"{sensor_id}.parquet"
        )
        df_joined.to_parquet(
            out_path,
            engine="pyarrow",
            compression=compression,
            index=False,
        )

        del df_sensor, df_joined
        gc.collect()

    # ------------------------------------------------------------------
    # ✅ SAFE STITCHING VIA PYARROW (NO DUCKDB UNION)
    # ------------------------------------------------------------------
    print("[2/3] Stitching chunks with PyArrow...")

    tables = []
    for f in sorted(os.listdir(chunk_dir)):
        if f.endswith(".parquet"):
            tables.append(pq.read_table(os.path.join(chunk_dir, f)))

    if not tables:
        raise RuntimeError("No chunk files produced.")

    final_table = pa.concat_tables(tables, promote=True)
    pq.write_table(final_table, output_file, compression=compression)

    print(f"[2/3] ✅ Output written: {output_file}")


# -----------------------------------------------------------------------------
# Step 3: Sanity checks
# -----------------------------------------------------------------------------

def run_sanity_checks(con, output_file):
    print("\n[3/3] Sanity checks...")

    future = con.execute(f"""
        SELECT COUNT(*) AS n_future
        FROM '{output_file}'
        WHERE weather_time > time
    """).df()

    age = con.execute(f"""
        SELECT
            MIN(minutes_since_weather_update) AS min_age,
            APPROX_QUANTILE(minutes_since_weather_update, 0.5) AS p50_age,
            APPROX_QUANTILE(minutes_since_weather_update, 0.95) AS p95_age,
            MAX(minutes_since_weather_update) AS max_age
        FROM '{output_file}'
    """).df()

    print("\nFuture leakage (should be 0):")
    print(future)

    print("\nWeather age distribution (minutes):")
    print(age)


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def main(cli_args=None):
    parser = argparse.ArgumentParser(allow_abbrev=False)

    parser.add_argument("--flood_file", required=True)
    parser.add_argument("--precip_master", required=True)
    parser.add_argument("--output_file", required=True)
    parser.add_argument("--chunk_dir", default="temp_chunks")
    parser.add_argument("--asof_tolerance_min", type=int, default=15)

    args, _ = parser.parse_known_args(cli_args)

    con = connect_duckdb()

    mapping_df = build_nearest_station_mapping(
        con,
        args.flood_file,
        args.precip_master,
    )

    temporal_join_per_sensor(
        con,
        args.flood_file,
        args.precip_master,
        mapping_df,
        args.output_file,
        args.chunk_dir,
        args.asof_tolerance_min,
    )

    run_sanity_checks(con, args.output_file)
    con.close()

    print("\n✅ DONE")


data_dir = PROJECT_ROOT / "Data_Files"

# Construct the argument list with absolute paths resolved at runtime
modeling_args = [
    "--flood_file", str(data_dir / "floodnet_full_dataset_merged.parquet"),
    "--precip_master", str(data_dir / "nyc_precipitation_master.parquet"),
    "--output_file", str(data_dir / "floodnet_full_dataset_merged_with_weather.parquet")
]

In [ ]:
if __name__ == "__main__":
    main(modeling_args)


[1/3] Extracting unique sensors...
[1/3] ✅ 304 sensors → 18 stations

[2/3] Loading precipitation master...
[2/3] Temporal join (304 sensors)
   [1] asleep_apricot_bedbug
   [25] evenly_alive_grouse
   [50] humbly_casual_heron
   [75] merely_equal_alien
   [100] possibly-positive-jawfish
   [125] solely_nice_beagle
   [150] yearly-hardy-sloth
   [175] early_many_oyster
   [200] highly_comic_akita
   [225] merely_upward_goblin
   [250] purely-finer-beagle
   [275] solely_strong_eft
